In [ ]:
import numpy as np
import math
import scipy
import random
import pandas as pd
import geopandas as gpd
import networkx as nx
import osmnx as ox
import shapely
from shapely.geometry import Point, MultiPoint, LineString, MultiLineString
from shapely.geometry.polygon import Polygon
from geopy.geocoders import Nominatim
import string
import colormaps as cmaps
import matplotlib as mpl
import matplotlib.pyplot as plt
from matplotlib import cm
from pyproj import Proj, transform

ox.settings.log_console = True
random.seed(42)

from mpl_toolkits.axes_grid1 import make_axes_locatable
plt.rcParams.update({
    "text.usetex": True,
    "font.family": 'STIXGeneral'
})
plt.rcParams['pdf.fonttype'] = 42
plt.rcParams['ps.fonttype'] = 42
plt.rcParams['svg.fonttype'] = 'none'
mpl.rcParams['figure.dpi'] = 600
mpl.rcParams['mathtext.fontset'] = 'stix'

plt.rcParams.update({'font.size': 14})

from archive.road_characteristics_imputation import preprocess_road_data

In [ ]:
# does the script need to produce the network edges, nodes, LSOA distances, OD list, and OD demand?
firstRun = True

In [ ]:
def ENtoLL(easting,northing):
    lat,lon = transform(Proj('epsg:27700'), Proj('epsg:4326'), easting, northing)
    return lon,lat

In [ ]:
### DEFINE CONSTANTS ###
year = 2019 # which OD data year to use
numDays = 365 # that's just a fact (don't talk to me about leap years)
tol = 58.6 # 85 # distance tolerance (m) for node merging = median LSOA radius in Sheffield
milesToKm = 1.60934 # multiplication factor for speed limits
avgSpeed = 17.7 # km per hour (estimated from ringroad sensors, using capacity/critical density)
congFactor = avgSpeed/(40*milesToKm) # congestion factor, using that the speed limit of the ringroad is 40mph
matlab = 1 # PLUS ONE TO THE INDICES FOR MATLAB'S SAKE!

mainDir = '' # INSERT WORKING DIRECTORY
imgOutDir = mainDir + 'plots/'
inputDir = mainDir + 'inputs/'
outputDir = mainDir + 'inputs/' # this remains as inputs, as the TA model reads from the inputs folder

# sensor data (processed)
sensor_path = inputDir + 'Data_Prepared_Density.csv'

# shape data
lsoa_path = inputDir + 'LSOA_2011_EW_BGC.zip' # TOO LARGE TO UPLOAD TO GITHUB - DOWNLOAD SHAPE FILES FROM OPEN GEOGRAPHY PORTAL (OPEN ACCESS)
sheff_path = inputDir + 'sheffield_roads.shp'

# lookups
lsoa_msoa_lookup_path = inputDir + 'LSOA_to_MSOA_2021.csv'  # TOO LARGE TO UPLOAD TO GITHUB - DOWNLOAD LOOKUP TABLE FROM OPEN GEOGRAPHY PORTAL (OPEN ACCESS)
cap_key_path = inputDir + 'TfL_capacities.csv'
road_class_key_path = inputDir + 'TfL_road_class_key_no_widths.csv'
lsoa_pwc_path = inputDir + 'LSOAs_DEC_2011_EW_PWC.csv'

bound_path = inputDir + 'Local_Authority_Dec_2024_Boundaries.zip'

# names of output files
# OD data
OD_path = outputDir + 'OD_matrix.csv'
OD_names_path = outputDir + 'OD_names.csv'
demand_path = outputDir + 'demand.csv'
# full network
G_edges_path = outputDir + 'edges_full.csv'
G_nodes_path = outputDir + 'nodes_full.csv'
G_soloNodes_path = outputDir + 'soloNodes_full.csv'
G_lDists_path = outputDir + 'lDists_full.csv'
G_OD_list_path = outputDir + 'OD_list_full.csv'
# reduced network
G2_edges_path = outputDir + 'edges.csv'
G2_nodes_path = outputDir + 'nodes.csv'
G2_soloNodes_path = outputDir + 'soloNodes_tol' + str(tol) + '.csv'
G2_lDists_path = outputDir + 'lDists_tol' + str(tol) + '.csv'
G2_OD_list_path = outputDir + 'OD_list_tol' + str(tol) + '.csv'

In [ ]:
### LOAD DATA ###
# sensor data (processed)
sensor = pd.read_csv(sensor_path)

# shape data
lsoas = gpd.read_file(lsoa_path).to_crs('WGS84')
sheff = gpd.read_file(sheff_path).to_crs('WGS84')
bound = gpd.read_file(bound_path).to_crs('WGS84')

# lookups
lsoa_msoa_lookup = pd.read_csv(lsoa_msoa_lookup_path)
cap_key = pd.read_csv(cap_key_path)
road_class_key = pd.read_csv(road_class_key_path)
# OA_areas = pd.read_csv(OA_areas_path)
# lane_key = pd.read_csv(lane_key_path)
lsoa_pwc = pd.read_csv(lsoa_pwc_path)

# OD data
AM_peak = []
midday = []
PM_peak = []
night = []
daily = []
monthRange = range(2,10)
for ii in monthRange: # read each month of data
    f1 = pd.read_csv(inputDir + 'OD_' + str(year) + '_' + str(ii) + '_WD_AM_PEAK_LSOA.csv') # time: 7-9 (2h)
    f2 = pd.read_csv(inputDir + 'OD_' + str(year) + '_' + str(ii) + '_WD_MIDDAY_LSOA.csv') # time: 10-15 (5h)
    f3 = pd.read_csv(inputDir + 'OD_' + str(year) + '_' + str(ii) + '_WD_PM_PEAK_LSOA.csv') # time: 16-19 (3h)
    f4 = pd.read_csv(inputDir + 'OD_' + str(year) + '_' + str(ii) + '_WD_NIGHT_LSOA.csv') # time: 19-6 (11h)
    f5 = pd.read_csv(inputDir + 'OD_' + str(year) + '_' + str(ii) + '_WD_DAILY_LSOA.csv') # time: 7-19 (12h)
    if ii == min(monthRange):
        OD_names = list(f1)
        AM_peak = np.array(f1.iloc[:, 1:])
        midday = np.array(f2.iloc[:, 1:])
        PM_peak = np.array(f3.iloc[:, 1:])
        night = np.array(f4.iloc[:, 1:])
        daily = np.array(f5.iloc[:, 1:])
    else:
        AM_peak = AM_peak + np.array(f1.iloc[:, 1:])
        midday = midday + np.array(f2.iloc[:, 1:])
        PM_peak = PM_peak + np.array(f3.iloc[:, 1:])
        night = night + np.array(f4.iloc[:, 1:])
        daily = daily + np.array(f5.iloc[:, 1:])
        if ii == max(monthRange): # average the demand over the number of months
            AM_peak = AM_peak / len(monthRange)
            midday = midday / len(monthRange)
            PM_peak = PM_peak / len(monthRange)
            night = night / len(monthRange)
            daily = daily / len(monthRange)

# load output files
if not firstRun:
    # OD data
    demand = pd.read_csv(demand_path)
    ODlist = pd.read_csv(G_OD_list_path)
    OD_names = pd.read_csv(OD_names_path).values.tolist()
    # full network
    G_edges = gpd.read_file(G_edges_path, GEOM_POSSIBLE_NAMES="geometry", KEEP_GEOM_COLUMNS="NO")
    G_nodes = gpd.read_file(G_nodes_path, GEOM_POSSIBLE_NAMES="geometry", KEEP_GEOM_COLUMNS="NO")
    lDists = pd.read_csv(G_lDists_path)
    # reduced network
    G2_edges = gpd.read_file(G2_edges_path, GEOM_POSSIBLE_NAMES="geometry", KEEP_GEOM_COLUMNS="NO")
    G2_nodes = gpd.read_file(G2_nodes_path, GEOM_POSSIBLE_NAMES="geometry", KEEP_GEOM_COLUMNS="NO")

In [ ]:
# get the boundary of Sheffield
bound = bound[bound.LAD24NM == 'Sheffield']
xy = bound.get_coordinates()

# filter LSOAs to Sheffield
lsoas = lsoas.loc[lsoas.LSOA11NM.str.contains('Sheffield')].reset_index(drop=True)

lon,lat = ENtoLL(lsoa_pwc['x'], lsoa_pwc['y'])
lsoa_pwc = lsoa_pwc.assign(x=lon, y=lat)

lsoas = lsoas.merge(lsoa_pwc[['LSOA11CD','x','y']])

In [ ]:
if firstRun:
    # CHOOSE WHICH TIMEBAND TO BECOME THE OD MATRIX
    OD_names = list(filter(lambda x: x != 'Unnamed: 0', OD_names))
    OD = pd.DataFrame(data=np.maximum(AM_peak, PM_peak))
    OD.columns = OD_names
    numHours = 2+3 # how many hours of data is in this timeband

    filt = list(filter(lambda x: x[1] != 'NOTINREGION', enumerate(OD_names)))
    inds = [row[0] for row in filt]
    OD_names = [row[1] for row in filt]
    OD = OD.loc[OD.index.isin(inds), OD_names]

    #OD.to_csv(OD_path)

In [ ]:
if firstRun:
    # Read & simplify South Yorkshire road network
    cf = '["highway"~"motorway|trunk|primary|secondary|tertiary|unclassified"]' # CAR NETWORK FOR PAPER
    G = ox.graph_from_place('Sheffield', custom_filter=cf, simplify=True)
    remove = [node for node, degree in dict(G.degree()).items() if degree < 2]
    G.remove_nodes_from(remove)

In [ ]:
if firstRun:
    # Convert nodes to a DataFrame
    G_nodes, G_edges = ox.graph_to_gdfs(G)
    G_edges = G_edges.to_crs('WGS84')

In [ ]:
if firstRun:
    # assign the multiindex u,v,key as separate columns, and reset index
    G_edges = G_edges.assign(u=G_edges.index.get_level_values(level=0).to_list(),
                            v=G_edges.index.get_level_values(level=1).to_list(),
                            key=G_edges.index.get_level_values(level=2).to_list()).reset_index(drop=True)

In [ ]:
if firstRun:
    # Ensure all values are lists
    G_edges['ID'] = G_edges['osmid'].apply(lambda x: list(x) if isinstance(x, (list, tuple)) else [x])
    # Convert list to tuple (to make them hashable)
    G_edges['ID_tuple'] = G_edges['ID'].apply(tuple)
    # Assign unique integer IDs
    G_edges['new_ID'] = pd.factorize(G_edges['ID_tuple'])[0]

    countJoiner = pd.DataFrame(G_edges['new_ID'].value_counts())
    countJoiner = countJoiner.assign(new_ID=countJoiner.index).reset_index(drop=True)
    G_edges = G_edges.merge(countJoiner, how='left')

In [ ]:
if firstRun:
    G_edges = G_edges.rename(columns={'lanes': 'lanes_OSMNx', 'width': 'width_OSMNx'})
    G_edges = G_edges.assign(speedlim=np.nan, lanes=np.nan, width=np.nan)
    for index, row in G_edges.iterrows():
        speed = row['maxspeed']
        if ~np.all(pd.isnull(speed)):
            if len(speed)>3:
                G_edges.loc[index,'speedlim'] = float(speed[:-4])
            else:
                if len(speed)==2:
                    speed0 = float(speed[0][:-4])
                    speed1 = float(speed[1][:-4])
                    G_edges.loc[index,'speedlim'] = max(speed0,speed1) #(speed0+speed1)/2
                else:
                    speed0 = float(speed[0][:-4])
                    speed1 = float(speed[1][:-4])
                    speed2 = float(speed[2][:-4])
                    G_edges.loc[index,'speedlim'] = max(speed0,speed1,speed2) #(speed0+speed1+speed2)/3
        lane = row['lanes_OSMNx']
        if ~np.all(pd.isnull(lane)):
            if len(lane)==1:
                G_edges.loc[index,'lanes'] = float(lane)
            else:
                lane0 = float(lane[0])
                lane1 = float(lane[1])
                G_edges.loc[index,'lanes'] = max(lane0,lane1)
                # if row['count'] > 1:
                #     G_edges.loc[index,'lanes'] = float(lane0)
                # else:
                #     G_edges.loc[index,'lanes'] = float(lane1)
        width = row['width_OSMNx']
        if ~np.all(pd.isnull(width)):
            if type(width)==str:
                G_edges.loc[index,'width'] = float(width)
            else:
                width = width[0]
                G_edges.loc[index,'width'] = float(width)
    # # infill missing values with defaults: lanes=2, speedlim=30
    # G_edges.loc[G_edges['lanes']<2, 'lanes'] = 2
    # G_edges.loc[G_edges['speedlim']<30, 'speedlim'] = 30

In [ ]:
# print(np.min(G_edges.loc[G_edges['oneway']==True, 'lanes']), np.max(G_edges.loc[G_edges['oneway']==True, 'lanes']),
#       np.min(G_edges.loc[G_edges['oneway']==False, 'lanes']), np.max(G_edges.loc[G_edges['oneway']==False, 'lanes']))

In [ ]:
# assign highway_key to deal with highways as lists
if firstRun:
    G_edges = G_edges.assign(highway_key = G_edges['highway'])
    for index, row in G_edges.iterrows():
        if len(row['highway_key']) == 2:
            G_edges.loc[index, 'highway_key'] = 'UT'

In [ ]:
# fig, ax = plt.subplots()

# G_edges.plot(
#     color="black",
#     ax=ax,
#     legend=False,
#     linewidth=0.1
# )
# G_edges.plot(
#     column="lanes",
#     ax=ax,
#     legend=True,
#     categorical=True,
#     linewidth=0.8,
#     cmap=mpl.cm.viridis_r
# )
# bound.plot(ax=ax, facecolor="none", edgecolor="black")

# plt.axis('off')
# ax.set_xlim([min(xy.x), max(xy.x)])
# ax.set_ylim([min(xy.y)-0.01, max(xy.y)])

# #plt.savefig(imgOutDir + 'Preprocessing/All_roads_lanes_pre_modelling.png', transparent=False, bbox_inches='tight')
# plt.show()

In [ ]:
# eccy = G_edges.loc[G_edges['name']==('Ecclesall Road')]#.reset_index(drop=True)
# eccy[['oneway','lanes_OSMNx','lanes','width']]

In [ ]:
# # Plot roads with 4 or more lanes
# fourLanes = G_edges.loc[G_edges['lanes']>=4]

# fig, ax = plt.subplots()

# G_edges.plot(
#     color="black",
#     ax=ax,
#     legend=False,
#     linewidth=0.1
# )
# fourLanes.plot(
#     column="lanes",
#     ax=ax,
#     legend=False,
#     linewidth=0.8,
#     cmap=mpl.cm.viridis
# )
# bound.plot(ax=ax, facecolor="none", edgecolor="black")

# plt.axis('off')
# ax.set_xlim([min(xy.x), max(xy.x)])
# ax.set_ylim([min(xy.y)-0.01, max(xy.y)])

# #plt.savefig(imgOutDir + 'Preprocessing/Roads_with_4_or_more_lanes.png', transparent=False, bbox_inches='tight')
# plt.show()

In [ ]:
#set(fourLanes['name'])

In [ ]:
#G_edges.loc[G_edges['name']=='Sheffield Parkway', ['highway','lanes_OSMNx','lanes','width','geometry']]

In [ ]:
#bloop=G_edges.loc[314,'oneway']

In [ ]:
#G_edges.loc[G_edges['oneway']==False]

In [ ]:
# abby = G_edges.loc[G_edges['name']==('Abbeydale Road')]#.reset_index(drop=True)
# print(set(abby['lanes']))

In [ ]:
# eccy = G_edges.loc[G_edges['name']==('Ecclesall Road')]#.reset_index(drop=True)
# print(set(eccy['lanes']))

In [ ]:
#len(eccy.loc[eccy['lanes']==2])

In [ ]:
#set(eccy.loc[eccy['lanes']==2,'osmid'])

In [ ]:
#eccy.loc[eccy['lanes']==4]

In [ ]:
# abbyTest = abby.loc[abby['new_ID']==1031]
# abbyTest[['geometry','lanes']]

In [ ]:
# # Plot Abbeydale Road
# fig, ax = plt.subplots()

# G_edges.plot(
#     color="black",
#     ax=ax,
#     legend=False,
#     linewidth=0.1
# )
# abbyTest.plot(
#     column="new_ID",
#     ax=ax,
#     legend=True,
#     #legend_kwds={"label": r'$\textrm{Number of lanes}$'},
#     categorical=True,
#     linewidth=0.6,
#     cmap=mpl.cm.viridis_r
# )
# bound.plot(ax=ax, facecolor="none", edgecolor="black")

# plt.axis('off')
# ax.set_xlim([min(xy.x), max(xy.x)])
# ax.set_ylim([min(xy.y)-0.01, max(xy.y)])

# #plt.savefig(imgOutDir + 'abbeydale_road_lanes.png', transparent=False, bbox_inches='tight')
# plt.show()

In [ ]:
#G_edges.loc[~np.isnan(G_edges['width']), ['new_ID','oneway','highway','name','speedlim','lanes','width']]

In [ ]:
# # use the above for calibration (manual)
# # Ecclesall Road
# G_edges.loc[(G_edges['name'] == 'Ecclesall Road') & (G_edges['lanes'] > 2), 'lanes'] = 2 # set all Ecclesall road to have max 2 lanes
# G_edges.loc[314, 'width'] = 6.29
# G_edges.loc[5034, 'width'] = 5.69
# G_edges.loc[3995, 'width'] = 3.1
# G_edges.loc[3995, 'lanes'] = 1 # only 1 lane here
# G_edges.loc[4004, 'width'] = 8.12
# G_edges.loc[(G_edges['name'] == 'Ecclesall Road') & (np.isnan(G_edges['width'])), 'width'] = 5.45 # set all remaining Abbeydale road to have 5.45m width
# # Abbeydale Road
# G_edges.loc[(G_edges['name'] == 'Abbeydale Road') & (G_edges['lanes'] > 3), 'lanes'] = 3 # set all Abbeydale road to have max 3 lanes
# G_edges.loc[4633, 'width'] = 5.8
# G_edges.loc[4633, 'lanes'] = 2
# G_edges.loc[5846, 'width'] = 4.38
# G_edges.loc[5846, 'lanes'] = 2
# G_edges.loc[(G_edges['name'] == 'Abbeydale Road') & (np.isnan(G_edges['width'])), 'width'] = 5.25 # set all remaining Abbeydale road to have 5.25m width
# # Fix width anomoly on Hollins Lane
# G_edges.loc[G_edges['name'] == 'Hollins Lane', 'width'] = 7 # from 1.98m to 7m
# # Sheffield Parkway
# G_edges.loc[G_edges['name'] == 'Sheffield Parkway', 'width'] = 11.3/3 * G_edges.loc[G_edges['name'] == 'Sheffield Parkway', 'lanes']


In [ ]:
# fig, ax = plt.subplots()

# G_edges.plot(
#     color="black",
#     ax=ax,
#     legend=False,
#     linewidth=0.1
# )
# G_edges.plot(
#     column="lanes",
#     ax=ax,
#     legend=True,
#     categorical=True,
#     linewidth=0.8,
#     cmap=mpl.cm.viridis_r
# )
# bound.plot(ax=ax, facecolor="none", edgecolor="black")

# plt.axis('off')
# ax.set_xlim([min(xy.x), max(xy.x)])
# ax.set_ylim([min(xy.y)-0.01, max(xy.y)])

# #plt.savefig(imgOutDir + 'Preprocessing/All_roads_lanes_mid_modelling.png', transparent=False, bbox_inches='tight')
# plt.show()

In [ ]:
#G_edges.loc[G_edges['name']=='Sheffield Parkway', ['highway','lanes_OSMNx','lanes','width','geometry']]

In [ ]:
# fig, ax = plt.subplots()

# G_edges.plot(
#     color="black",
#     ax=ax,
#     legend=False,
#     linewidth=0.1
# )
# G_edges.plot(
#     column="lanes",
#     ax=ax,
#     legend=True,
#     categorical=True,
#     linewidth=0.8,
#     cmap=mpl.cm.viridis_r
# )
# bound.plot(ax=ax, facecolor="none", edgecolor="black")

# plt.axis('off')
# ax.set_xlim([min(xy.x), max(xy.x)])
# ax.set_ylim([min(xy.y)-0.01, max(xy.y)])

# #plt.savefig(imgOutDir + 'Preprocessing/All_roads_lanes.png', transparent=False, bbox_inches='tight')
# plt.show()

In [ ]:
# use function to infill missing speedlim, lanes, width
if firstRun:
    G_edges = preprocess_road_data(G_edges)
    # data = G_edges[['highway_key','lanes','speedlim','width']].drop_duplicates()
    # #data.to_csv(mainDir + 'highway_check_width.csv')
    # data = preprocess_road_data(data)
    # # join to G_edges
    # G_edges = G_edges.drop(columns=['lanes','speedlim','width'])
    # G_edges = G_edges.merge(data, how='left')

In [ ]:
fig, ax = plt.subplots()

G_edges.plot(
    color="black",
    ax=ax,
    legend=False,
    linewidth=0.1
)
G_edges.plot(
    column="lanes",
    ax=ax,
    legend=True,
    categorical=True,
    linewidth=0.8,
    cmap=mpl.cm.viridis_r
)
bound.plot(ax=ax, facecolor="none", edgecolor="black")

plt.axis('off')
ax.set_xlim([min(xy.x), max(xy.x)])
ax.set_ylim([min(xy.y)-0.01, max(xy.y)])

plt.savefig(imgOutDir + 'Preprocessing/All_roads_lanes_post_modelling.png', transparent=False, bbox_inches='tight')
plt.show()

In [ ]:
# fig, ax = plt.subplots()

# G_edges.plot(
#     color="black",
#     ax=ax,
#     legend=False,
#     linewidth=0.1
# )
# G_edges.plot(
#     column="width",
#     ax=ax,
#     legend=True,
#     categorical=False,
#     linewidth=0.8,
#     cmap=mpl.cm.viridis_r
# )
# bound.plot(ax=ax, facecolor="none", edgecolor="black")

# plt.axis('off')
# ax.set_xlim([min(xy.x), max(xy.x)])
# ax.set_ylim([min(xy.y)-0.01, max(xy.y)])

# plt.savefig(imgOutDir + 'Preprocessing/All_roads_width.png', transparent=False, bbox_inches='tight')
# plt.show()

In [ ]:
# # if speedlim AND lanes are both missing,
# # infill missing values with median per highway
# if firstRun:
#     speedlim_fill = G_edges[['highway_key','speedlim']].groupby('highway_key').transform(lambda x: x.fillna(x.median()))
#     lanes_fill = G_edges[['highway_key','lanes']].groupby('highway_key').transform(lambda x: x.fillna(x.median()))
#     for index, row in G_edges.iterrows():
#         if (np.isnan(row['speedlim'])) & (np.isnan(row['lanes'])):
#             G_edges.loc[index, 'speedlim'] = speedlim_fill.loc[0, 'speedlim']
#             G_edges.loc[index, 'lanes'] = lanes_fill.loc[0, 'lanes']

In [ ]:
# # if either speedlim OR lanes are missing,
# # infill missing values with median per highway and lanes OR speedlim
# if firstRun:
#     speedlim_fill = G_edges[['highway_key','lanes','speedlim']].groupby(['highway_key','lanes']).transform(lambda x: x.fillna(x.median()))
#     lanes_fill = G_edges[['highway_key','lanes','speedlim']].groupby(['highway_key','speedlim']).transform(lambda x: x.fillna(x.median()))
#     for index, row in G_edges.iterrows():
#         if np.isnan(row['speedlim']):
#             G_edges.loc[index, 'speedlim'] = speedlim_fill.loc[0, 'speedlim']
#         if np.isnan(row['lanes']):
#             G_edges.loc[index, 'lanes'] = lanes_fill.loc[0, 'lanes']
#     # if any are still missing, infill with total median
#     G_edges['speedlim'] = G_edges['speedlim'].fillna(np.median(G_edges['speedlim']))
#     G_edges['lanes'] = G_edges['lanes'].fillna(np.median(G_edges['lanes']))

In [ ]:
# width_check = G_edges[['highway_key','speedlim','lanes']].drop_duplicates()
# width_check
# width_check.to_csv('width_check.csv')

In [ ]:
# # using the above model,
# # join width estimates
# if firstRun:
#     G_edges = G_edges.merge(result, how='left')
#     width_check = G_edges.loc[~np.isnan(G_edges['width']), ['width','estimated_width_m']].drop_duplicates()
#     #width_check.to_csv('width_check.csv')

In [ ]:
if firstRun:
    #road_class_key = road_class_key.drop(columns=['width'])
    G_edges = G_edges.merge(road_class_key, how='left', left_on=['highway_key','lanes','speedlim'], right_on=['highway_key','lanes','speedlim'])

In [ ]:
#missing = G_edges.loc[~(G_edges['class'].isin(['UAP1', 'UAP2', 'UAP3', 'UAP4'])), ['highway_key','lanes','speedlim']].drop_duplicates()
#missing
#missing.to_csv('missing.csv')

In [ ]:
# use non-linear regression output (in R) to fit capacities,
# based on the TfL lookup table
if firstRun:
    G_edges = G_edges.assign(road_class = 1)
    G_edges.loc[G_edges['class']=='UAP2', 'road_class'] = 2
    G_edges.loc[G_edges['class']=='UAP3', 'road_class'] = 3
    G_edges.loc[G_edges['class']=='UAP4', 'road_class'] = 4
    ra=274.29825
    rb=-0.07768
    rc=0.90550
    rd=-0.30351
    G_edges = G_edges.assign(capacity = ra * (G_edges['lanes']**rb) * (G_edges['width']**rc) * (G_edges['road_class']**rd))

In [ ]:
# if firstRun:
#     highways = G_edges['highway']
#     G_edges = G_edges.merge(road_class_key, how='left', left_on=['highway_key','lanes','speedlim'], right_on=['highway_key','lanes','speedlim'])
#     G_edges = G_edges.merge(cap_key, how='left', left_on=['class','lanes','width'], right_on=['class','lanes','width'])

In [ ]:
# if firstRun:
    # G_edges_missing_capacity = G_edges.loc[np.isnan(G_edges['capacity'])]
    # G_edges_missing_capacity[['highway','highway_label','speedlim','lanes']].drop_duplicates().reset_index(drop=True)

In [ ]:
# if firstRun:
#     # infill missing capacities with default = 1110
#     G_edges.loc[~np.isfinite(G_edges['capacity']), 'capacity'] = 1110

In [ ]:
if firstRun:
    # TfL lookup table assumes 60/40 flow split; capacities are for busiest direction
    # This is unknown information, so choose a 50/50 split
    G_edges['capacity'] = G_edges['capacity'] * 5/6

In [ ]:
if firstRun:
    # assign critical density: road capacity (veh/time) divided by speedlim (km/time)
    # capacity: veh/hour
    # speedlim: miles/hour -> km/hour
    # criticalDensity: veh/km
    G_edges['speedlim'] = G_edges['speedlim'] * milesToKm
    # assign congestion factor from ringroad
    G_edges = G_edges.assign(congFactor = round(congFactor,1))
    # on motorways, allow twice the fraction
    G_edges.loc[G_edges['highway'].isin(['motorway','motorway_link']), 'congFactor'] = 2*round(congFactor,1)
    # assign average speed as speedlim * congFactor
    G_edges = G_edges.assign(avgSpeed = np.multiply(G_edges['speedlim'], G_edges['congFactor']))
    # assign critical density as capacity/average speed
    G_edges = G_edges.assign(criticalDensity = np.divide(G_edges['capacity'], G_edges['avgSpeed']))

In [ ]:
if firstRun:
    G_nodes = G_nodes.assign(LSOA="", LSOA_Lon="", LSOA_Lat="", PWC_Distance="")
    for index, row in G_nodes.iterrows():
        pnt = row['geometry']
        matches = np.zeros([len(lsoas),1])
        for ind, lsoa in lsoas.iterrows():
            polygon = lsoa.geometry
            if polygon.contains(pnt):
                matches[ind] = 1
                hit = np.where(matches)[0][0]
        dist_to_pwc = math.dist([row['x'], row['y']], [lsoas.loc[hit, 'x'],lsoas.loc[hit, 'y']])
        G_nodes.loc[index, 'LSOA'] = lsoas.loc[hit, 'LSOA11CD']
        G_nodes.loc[index, 'LSOA_Lon'] = lsoas.loc[hit, 'x']
        G_nodes.loc[index, 'LSOA_Lat'] = lsoas.loc[hit, 'y']
        G_nodes.loc[index, 'PWC_Distance'] = dist_to_pwc

In [ ]:
if firstRun:
    # ID from 1:n
    G_nodes = G_nodes.assign(NodeID=np.arange(1,len(G_nodes)+1))

In [ ]:
if firstRun:
    nodeKey = G_nodes[['LSOA','PWC_Distance','NodeID','x','y']].rename(columns={'x': 'nodeX', 'y': 'nodeY'})

In [ ]:
if firstRun:    
    # join node info to LSOAs
    lDists = pd.merge(lsoas, nodeKey, how='left', left_on='LSOA11CD', right_on='LSOA')

    # get nodeless LSOAs
    nodeless = lDists.loc[np.isnan(lDists['NodeID'])]

    # get single node LSOAs
    singleNodeLSOAs = lDists.groupby('LSOA').count()
    singleNodeLSOAs = singleNodeLSOAs.loc[singleNodeLSOAs['NodeID']==1].index.values

In [ ]:
if firstRun:
    soloNodes1 = pd.DataFrame(singleNodeLSOAs).rename(columns={0: 'LSOA'}).assign(Solo = 1)
    soloNodes2 = pd.DataFrame(nodeless['LSOA11CD'].values).rename(columns={0: 'LSOA'}).assign(Nodeless = 1)
    soloNodes = soloNodes1.merge(soloNodes2, how='outer', on='LSOA')
    soloNodes = soloNodes.fillna(0)

In [ ]:
if firstRun:
    # compute distance matrix between nodeless LSOA PWCs and all node coordinates
    dm = scipy.spatial.distance_matrix(np.array([nodeless['x'],nodeless['y']]).T,
                                    np.array([G_nodes['x'],G_nodes['y']]).T)

In [ ]:
if firstRun:
    # ensure no LSOAs are nodeless
    lDists = lDists.assign(Nodeless=0) # flag the nodeless LSOAs that have been nearest joined
    for index in range(np.shape(dm)[0]):
        row = dm[index,]
        minDist = min(row)
        minIdx = np.where(row==min(row))[0][0]
        lDists.loc[lDists.index==nodeless.index[index], 'LSOA'] = lDists.loc[lDists.index==nodeless.index[index], 'LSOA11CD']
        lDists.loc[lDists.index==nodeless.index[index], 'NodeID'] = G_nodes.loc[G_nodes.index == G_nodes.index[minIdx], 'NodeID'].values[0]
        lDists.loc[lDists.index==nodeless.index[index], 'nodeX'] = G_nodes.loc[G_nodes.index == G_nodes.index[minIdx], 'x'].values[0]
        lDists.loc[lDists.index==nodeless.index[index], 'nodeY'] = G_nodes.loc[G_nodes.index == G_nodes.index[minIdx], 'y'].values[0]
        lDists.loc[lDists.index==nodeless.index[index], 'PWC_Distance'] = minDist
        lDists.loc[lDists.index==nodeless.index[index], 'Nodeless'] = 1

In [ ]:
# OD DEMAND #

In [ ]:
if firstRun:    
    ODmat = np.array(OD)
    totalDemand = np.sum(ODmat)
    # how much of the demand is intra-LSOA
    diagDemand = np.trace(ODmat)
    diagDemandPrcnt = diagDemand/totalDemand*100

In [ ]:
if firstRun:
    # retain OD data corresponding to the Sheffield LSOAs
    inds = np.where(pd.Series(OD_names).isin(lsoas['LSOA11CD']))[0]
    OD_names = [OD_names[i] for i in inds]
    OD = OD.loc[OD.index.isin(inds),OD_names].reset_index(drop=True)
    ODmat = ODmat[np.ix_(inds,inds)]

In [ ]:
if firstRun:
    # Get rows and columns of non-zero demand
    r = []
    c = []
    for ii in range(np.shape(ODmat)[0]):
        for jj in range(np.shape(ODmat)[0]):
            if ODmat[ii,jj] != 0:
                r.append(ii)
                c.append(jj)

In [ ]:
if firstRun:
    # get vectorised non-zero demand
    demand = ODmat[r,c]

In [ ]:
if firstRun:
    n = len(G_nodes)
    r = [x+(matlab) for x in r]
    c = [x+(matlab) for x in c]

In [ ]:
if firstRun:
    ODlist = np.column_stack((r,c))

In [ ]:
if firstRun:    
    # Probabilistically assign nodes as ODs based on their LSOA PWC distance
    nodesO = np.zeros([len(ODlist),1])
    nodesD = np.zeros([len(ODlist),1])
    for index in range(len(ODlist)):
        lsoaO = OD_names[ODlist[index,0]-(matlab)]
        lsoaD = OD_names[ODlist[index,1]-(matlab)]
        hitsO = lDists.loc[lDists['LSOA']==lsoaO]
        hitsD = lDists.loc[lDists['LSOA']==lsoaD]
        hitsO = hitsO.assign(Prob=np.divide(1/hitsO['PWC_Distance'], sum(1/hitsO['PWC_Distance'])))
        hitsD = hitsD.assign(Prob=np.divide(1/hitsD['PWC_Distance'], sum(1/hitsD['PWC_Distance'])))
        hitsO = hitsO.sort_values('Prob')
        hitsD = hitsD.sort_values('Prob')
        probO = random.random()
        probD = random.random()
        nodesO[index] = hitsO.loc[hitsO.index == hitsO.index[np.where(probO <= np.cumsum(hitsO['Prob']))[0][0]], 'NodeID'].values[0].astype('int')
        nodesD[index] = hitsD.loc[hitsD.index == hitsD.index[np.where(probD <= np.cumsum(hitsD['Prob']))[0][0]], 'NodeID'].values[0].astype('int')

In [ ]:
if firstRun:
    # Add the OD nodes to the ODlist
    ODlist = np.append(ODlist, nodesO, axis=1)
    ODlist = np.append(ODlist, nodesD, axis=1)

In [ ]:
if firstRun:
    # WRITE OUTPUTS #
    # OD data
    pd.DataFrame(demand).to_csv(demand_path, index=False)
    pd.DataFrame(ODmat).to_csv(OD_path, index=False)
    pd.DataFrame(OD_names).to_csv(OD_names_path, index=False)
    # full network
    G_edges.to_csv(G_edges_path)
    G_nodes.to_csv(G_nodes_path)
    lDists.to_csv(G_lDists_path, index=False)
    soloNodes.to_csv(G_soloNodes_path, index=False)
    pd.DataFrame(ODlist).to_csv(G_OD_list_path, index=False)

In [ ]:
# RECONSTRUCT GRAPH #

In [ ]:
#firstRun = True

In [ ]:
if firstRun:
    G = ox.graph_from_gdfs(G_nodes, G_edges.set_index(['u','v','key']), graph_attrs={'crs':'WGS84'})

In [ ]:
if firstRun:
    # get a GeoSeries of consolidated intersections
    G_proj = ox.projection.project_graph(G)#, to_crs='WGS84')
    ints = ox.simplification.consolidate_intersections(
        G_proj, rebuild_graph=False, tolerance=tol, dead_ends=False
    )
    print(len(ints))

In [ ]:
if firstRun:
    # compare to number of nodes in original graph
    print(len(G))

In [ ]:
if firstRun:
    # consolidate intersections and rebuild graph topology
    # this reconnects edge geometries to the new consolidated nodes
    G2 = ox.simplification.consolidate_intersections(
        G_proj, rebuild_graph=True, tolerance=tol, dead_ends=False
    )
    remove = [node for node, degree in dict(G2.degree()).items() if degree < 2]
    G2.remove_nodes_from(remove)
    print(len(G2))

In [ ]:
if firstRun:
    connectedNodes = [c for c in nx.strongly_connected_components(G2)]
    disconnectedNodes = np.array([connectedNodes[0].pop(), connectedNodes[1].pop(), connectedNodes[2].pop(),
                                connectedNodes[4].pop(), connectedNodes[5].pop(), connectedNodes[6].pop()])
    # ordered node IDs of disconnected components
    disconnectedNodes = np.sort(disconnectedNodes) + 1

In [ ]:
if firstRun:
    G2 = ox.utils_graph.get_largest_component(G2, strongly=True)
    print(len(G2))

    # Convert nodes to a DataFrame
    G2_nodes, G2_edges = ox.graph_to_gdfs(G2)

    # assign new node IDs
    G2_nodes = G2_nodes.assign(osmid_Old = G2_nodes.index, osmid = range(0,len(G2_nodes)))
    G2_nodes = G2_nodes.assign(NodeID_New = G2_nodes['osmid'])
    G2_nodes = G2_nodes.set_index('osmid')

In [ ]:
if firstRun:
    G2_edges = G2_edges.assign(idx = G2_edges.index)
    G2_edges = G2_edges.reset_index(drop=True)
    
    G2_edges = G2_edges.assign(u_Old=0, v_Old=0, key_Old=0)
    for index, row in G2_edges.iterrows():
        G2_edges.loc[index, 'u_Old'] = row['idx'][0]
        G2_edges.loc[index, 'v_Old'] = row['idx'][1]
        G2_edges.loc[index, 'key_Old'] = row['idx'][2]

    G2_edges = G2_edges.assign(u=0, v=0, key=0)
    for index, row in G2_edges.iterrows():
        s = row['u_Old']
        t = row['v_Old']
        sIdx = G2_nodes.loc[G2_nodes['osmid_Old']==s, 'NodeID_New'].values[0]
        tIdx = G2_nodes.loc[G2_nodes['osmid_Old']==t, 'NodeID_New'].values[0]
        G2_edges.loc[index, 'u'] = sIdx
        G2_edges.loc[index, 'v'] = tIdx

In [ ]:
if firstRun:
    # # Convert nodes to a DataFrame
    # G2_nodes, G2_edges = ox.graph_to_gdfs(G2)
    
    G2_nodes = G2_nodes.to_crs('WGS84')
    G2_edges = G2_edges.to_crs('WGS84')

    coords = pd.DataFrame(shapely.get_coordinates(G2_nodes['geometry']))
    G2_nodes.x = coords[0].values
    G2_nodes.y = coords[1].values

In [ ]:
if firstRun:
    G2 = ox.graph_from_gdfs(G2_nodes, G2_edges.set_index(['u','v','key']), graph_attrs={'crs':'WGS84'})
    # Convert nodes to a DataFrame
    G2_nodes, G2_edges = ox.graph_to_gdfs(G2)

In [ ]:
# REASSIGN LSOAS AND CREATE LDISTS2 #

In [ ]:
if firstRun:
    # ID
    G2_nodes = G2_nodes.assign(NodeID=np.arange(1,len(G2_nodes)+1))
    # blank values post-merging
    nodefill = G2_nodes.loc[np.isnan(G2_nodes['PWC_Distance'])]
    G2_nodes_pop = G2_nodes.loc[~np.isnan(G2_nodes['PWC_Distance'])]

In [ ]:
if firstRun:
    nodefill = nodefill.assign(LSOA="", LSOA_Lon="", LSOA_Lat="", PWC_Distance="")
    for index, row in nodefill.iterrows():
        pnt = row['geometry']
        matches = np.zeros([len(lsoas),1])
        for ind, lsoa in lsoas.iterrows():
            polygon = lsoa.geometry
            if polygon.contains(pnt):
                matches[ind] = 1
                hit = np.where(matches)[0][0]
        dist_to_pwc = math.dist([row['x'], row['y']], [lsoas.loc[hit, 'x'],lsoas.loc[hit, 'y']])
        nodefill.loc[index, 'LSOA'] = lsoas.loc[hit, 'LSOA11CD']
        nodefill.loc[index, 'LSOA_Lon'] = lsoas.loc[hit, 'x']
        nodefill.loc[index, 'LSOA_Lat'] = lsoas.loc[hit, 'y']
        nodefill.loc[index, 'PWC_Distance'] = dist_to_pwc

In [ ]:
if firstRun:    
    G2_nodes = pd.concat([G2_nodes_pop, nodefill])
    G2_nodes = G2_nodes.assign(NodeID=G2_nodes.index)
    G2_nodes = G2_nodes.sort_values(by='NodeID', ascending=True)

In [ ]:
if firstRun:
    nodeKey = G2_nodes[['LSOA','PWC_Distance','NodeID','x','y']].rename(columns={'x': 'nodeX', 'y': 'nodeY'})

In [ ]:
if firstRun:    
    # join node info to LSOAs
    lDists2 = pd.merge(lsoas, nodeKey, how='left', left_on='LSOA11CD', right_on='LSOA')

    # get nodeless LSOAs
    nodeless = lDists2.loc[np.isnan(lDists2['NodeID'])]

    # get single node LSOAs
    singleNodeLSOAs = lDists2.groupby('LSOA').count()
    singleNodeLSOAs = singleNodeLSOAs.loc[singleNodeLSOAs['NodeID']==1].index.values

In [ ]:
if firstRun:
    soloNodes1 = pd.DataFrame(singleNodeLSOAs).rename(columns={0: 'LSOA'}).assign(Solo = 1)
    soloNodes2 = pd.DataFrame(nodeless['LSOA11CD'].values).rename(columns={0: 'LSOA'}).assign(Nodeless = 1)
    soloNodes = soloNodes1.merge(soloNodes2, how='outer', on='LSOA')
    soloNodes = soloNodes.fillna(0)

In [ ]:
if firstRun:
    # compute distance matrix between nodeless LSOA PWCs and all node coordinates
    dm = scipy.spatial.distance_matrix(np.array([nodeless['x'],nodeless['y']]).T,
                                    np.array([G2_nodes['x'],G2_nodes['y']]).T)

In [ ]:
if firstRun:
    # ensure no LSOAs are nodeless
    lDists2 = lDists2.assign(Nodeless=0) # flag the nodeless LSOAs that have been nearest joined
    for index in range(np.shape(dm)[0]):
        row = dm[index,]
        minDist = min(row)
        minIdx = np.where(row==min(row))[0][0]
        lDists2.loc[lDists2.index==nodeless.index[index], 'LSOA'] = lDists2.loc[lDists2.index==nodeless.index[index], 'LSOA11CD']
        lDists2.loc[lDists2.index==nodeless.index[index], 'NodeID'] = G2_nodes.loc[G2_nodes.index == G2_nodes.index[minIdx], 'NodeID'].values[0]
        lDists2.loc[lDists2.index==nodeless.index[index], 'nodeX'] = G2_nodes.loc[G2_nodes.index == G2_nodes.index[minIdx], 'x'].values[0]
        lDists2.loc[lDists2.index==nodeless.index[index], 'nodeY'] = G2_nodes.loc[G2_nodes.index == G2_nodes.index[minIdx], 'y'].values[0]
        lDists2.loc[lDists2.index==nodeless.index[index], 'PWC_Distance'] = minDist
        lDists2.loc[lDists2.index==nodeless.index[index], 'Nodeless'] = 1

In [ ]:
# RECOMPUTE OD NODES #

In [ ]:
if firstRun:
    # select first two columns, i.e. the OD nodes
    ODlist2 = np.array(ODlist[:,0:2])
    #ODlist2[:,0:1] = ODlist2[:,0:1].astype('int')

In [ ]:
if firstRun:    
    # Probabilistically assign nodes as ODs based on their LSOA PWC distance
    nodesO = np.zeros([len(ODlist2),1])
    nodesD = np.zeros([len(ODlist2),1])
    for index in range(len(ODlist2)):
        lsoaO = OD_names[ODlist2[index,0].astype('int')-matlab]
        lsoaD = OD_names[ODlist2[index,1].astype('int')-matlab]
        hitsO = lDists2.loc[lDists2['LSOA']==lsoaO]
        hitsD = lDists2.loc[lDists2['LSOA']==lsoaD]
        hitsO = hitsO.assign(Prob=np.divide(1/hitsO['PWC_Distance'].astype('float'), sum(1/hitsO['PWC_Distance'].astype('float'))))
        hitsD = hitsD.assign(Prob=np.divide(1/hitsD['PWC_Distance'].astype('float'), sum(1/hitsD['PWC_Distance'].astype('float'))))
        hitsO = hitsO.sort_values('Prob')
        hitsD = hitsD.sort_values('Prob')
        probO = random.random()
        probD = random.random()
        nodesO[index] = hitsO.loc[hitsO.index == hitsO.index[np.where(probO <= np.cumsum(hitsO['Prob']))[0][0]], 'NodeID'].values[0].astype('int')
        nodesD[index] = hitsD.loc[hitsD.index == hitsD.index[np.where(probD <= np.cumsum(hitsD['Prob']))[0][0]], 'NodeID'].values[0].astype('int')

In [ ]:
if firstRun:
    # Add the OD nodes to the ODlist
    ODlist2 = np.append(ODlist2, nodesO, axis=1)
    ODlist2 = np.append(ODlist2, nodesD, axis=1)

In [ ]:
if firstRun:
    # WRITE OUTPUTS #
    G2_edges.to_csv(G2_edges_path)
    G2_nodes.to_csv(G2_nodes_path)
    soloNodes.to_csv(G2_soloNodes_path, index=False)
    lDists2.to_csv(G2_lDists_path, index=False)
    pd.DataFrame(ODlist2).to_csv(G2_OD_list_path, index=False)

In [ ]:
if not firstRun:
    G = ox.graph_from_gdfs(G_nodes, G_edges.set_index(['u','v','key']), graph_attrs={'crs':'WGS84'})
    G2_edges = G2_edges.set_index(['u','v','key'])

# reconstruct simplified graph with correct CRS
G2 = ox.graph_from_gdfs(G2_nodes, G2_edges, graph_attrs={'crs':'WGS84'})

In [ ]:
# Plot graph
fig, ax = plt.subplots()

bound.plot(ax=ax, facecolor="none", edgecolor="black")
ox.plot_graph(
    G,
    ax=ax,
    show=False,
    close=False,
    bgcolor="w",
    edge_color="black",
    edge_linewidth=0.3,#thicknesses,
    node_size=0
)

plt.axis('off')
ax.set_xlim([min(xy.x), max(xy.x)])
ax.set_ylim([min(xy.y)-0.01, max(xy.y)])

plt.savefig(imgOutDir + 'Reconstructed_Graph_Full.png', transparent=False, bbox_inches='tight')
plt.show()

In [ ]:
# Plot reduced OSMNx graph
fig, ax = plt.subplots()

bound.plot(ax=ax, facecolor="none", edgecolor="black")
ox.plot_graph(
    G2,
    ax=ax,
    show=False,
    close=False,
    bgcolor="w",
    edge_color="black",
    edge_linewidth=0.3,#thicknesses,
    node_size=0
)

plt.axis('off')
ax.set_xlim([min(xy.x), max(xy.x)])
ax.set_ylim([min(xy.y)-0.01, max(xy.y)])

plt.savefig(imgOutDir + 'Reconstructed_Graph_Tol' + str(tol) + '.png', transparent=False, bbox_inches='tight')
plt.show()

In [ ]:
G_edges.loc[G_edges['criticalDensity']=='', 'criticalDensity'] = '0'
G2_edges.loc[G2_edges['criticalDensity']=='', 'criticalDensity'] = '0'

In [ ]:
G_edges['criticalDensity'] = G_edges['criticalDensity'].astype('float')
G2_edges['criticalDensity'] = G2_edges['criticalDensity'].astype('float')

In [ ]:
G_edges.loc[G_edges['criticalDensity']==0, 'criticalDensity'] = np.mean(G_edges.loc[G_edges['criticalDensity'] !=0, 'criticalDensity'])
G2_edges.loc[G2_edges['criticalDensity']==0, 'criticalDensity'] = np.mean(G2_edges.loc[G2_edges['criticalDensity'] !=0, 'criticalDensity'])

In [ ]:
vMin = 0
vMax = max(max(G_edges['criticalDensity']), max(G2_edges['criticalDensity']))

In [ ]:
# Plot full OSMNx graph coloured by critical density
fig, ax = plt.subplots()

G_edges.plot(
    column="criticalDensity",
    ax=ax,
    legend=True,
    legend_kwds={"label": r'$\textrm{Critical density (veh/km)}$'},
    categorical=False,
    linewidth=0.5,
    cmap=mpl.cm.viridis_r
    # vmin=vMin,
    # vmax=vMax
)
bound.plot(ax=ax, facecolor="none", edgecolor="black")

plt.axis('off')
ax.set_xlim([min(xy.x), max(xy.x)])
ax.set_ylim([min(xy.y)-0.01, max(xy.y)])

plt.savefig(imgOutDir + 'Processed_Graph_critDens.png', transparent=False, bbox_inches='tight')
plt.show()

In [ ]:
# Plot reduced OSMNx graph coloured by critical density
fig, ax = plt.subplots()

G2_edges.plot(
    column="criticalDensity",
    ax=ax,
    legend=True,
    legend_kwds={"label": r'$\textrm{Critical density (veh/km)}$'},
    categorical=False,
    linewidth=0.5,
    cmap=mpl.cm.viridis_r
    # vmin=vMin,
    # vmax=vMax
)
bound.plot(ax=ax, facecolor="none", edgecolor="black")

plt.axis('off')
ax.set_xlim([min(xy.x), max(xy.x)])
ax.set_ylim([min(xy.y)-0.01, max(xy.y)])

plt.savefig(imgOutDir + 'Reconstructed_Graph_Tol' + str(tol) + '_critDens.png', transparent=False, bbox_inches='tight')
plt.show()

In [ ]:
G_edges['capacity'] = G_edges['capacity'].astype('float')
G2_edges['capacity'] = G2_edges['capacity'].astype('float')

In [ ]:
vMin = 0
vMax = max(max(G_edges['capacity']), max(G2_edges['capacity']))

In [ ]:
# Plot full OSMNx graph coloured by capacity
fig, ax = plt.subplots()

G_edges.plot(
    column="capacity",
    ax=ax,
    legend=True,
    legend_kwds={"label": r'$\textrm{Capacity (veh/hour)}$'},
    categorical=False,
    linewidth=0.5,
    cmap=mpl.cm.viridis_r
    # vmin=vMin,
    # vmax=vMax
)
bound.plot(ax=ax, facecolor="none", edgecolor="black")

plt.axis('off')
ax.set_xlim([min(xy.x), max(xy.x)])
ax.set_ylim([min(xy.y)-0.01, max(xy.y)])

plt.savefig(imgOutDir + 'Processed_Graph_capacity.png', transparent=False, bbox_inches='tight')
plt.show()

In [ ]:
# Plot reduced OSMNx graph coloured by capacity
fig, ax = plt.subplots()

G2_edges.plot(
    column="capacity",
    ax=ax,
    legend=True,
    legend_kwds={"label": r'$\textrm{Capacity (veh/hour)}$'},
    categorical=False,
    linewidth=0.5,
    cmap=mpl.cm.viridis_r
    # vmin=vMin,
    # vmax=vMax
)
bound.plot(ax=ax, facecolor="none", edgecolor="black")

plt.axis('off')
ax.set_xlim([min(xy.x), max(xy.x)])
ax.set_ylim([min(xy.y)-0.01, max(xy.y)])

plt.savefig(imgOutDir + 'Reconstructed_Graph_Tol' + str(tol) + '_capacity.png', transparent=False, bbox_inches='tight')
plt.show()

In [ ]:
# Plot full OSMNx graph coloured by width
fig, ax = plt.subplots()

G_edges.plot(
    column="width",
    ax=ax,
    legend=True,
    legend_kwds={"label": r'$\textrm{Road width (m)}$'},
    categorical=False,
    linewidth=0.5,
    cmap=mpl.cm.viridis_r
)
bound.plot(ax=ax, facecolor="none", edgecolor="black")

plt.axis('off')
ax.set_xlim([min(xy.x), max(xy.x)])
ax.set_ylim([min(xy.y)-0.01, max(xy.y)])

plt.savefig(imgOutDir + 'Full_Graph_width.png', transparent=False, bbox_inches='tight')
plt.show()

In [ ]:
# Plot reduced OSMNx graph coloured by width
fig, ax = plt.subplots()

G2_edges.plot(
    column="width",
    ax=ax,
    legend=True,
    legend_kwds={"label": r'$\textrm{Road width (m)}$'},
    categorical=False,
    linewidth=0.5,
    cmap=mpl.cm.viridis_r
)
bound.plot(ax=ax, facecolor="none", edgecolor="black")

plt.axis('off')
ax.set_xlim([min(xy.x), max(xy.x)])
ax.set_ylim([min(xy.y)-0.01, max(xy.y)])

plt.savefig(imgOutDir + 'Reconstructed_Graph_Tol' + str(tol) + '_width.png', transparent=False, bbox_inches='tight')
plt.show()

In [ ]:
# Plot reduced OSMNx graph coloured by lanes
fig, ax = plt.subplots()

G2_edges.plot(
    column="lanes",
    ax=ax,
    legend=True,
    #legend_kwds={"label": r'$\textrm{Number of lanes}$'},
    categorical=True,
    linewidth=0.5,
    cmap=mpl.cm.viridis_r
)
bound.plot(ax=ax, facecolor="none", edgecolor="black")

plt.axis('off')
ax.set_xlim([min(xy.x), max(xy.x)])
ax.set_ylim([min(xy.y)-0.01, max(xy.y)])

plt.savefig(imgOutDir + 'Reconstructed_Graph_Tol' + str(tol) + '_lanes.png', transparent=False, bbox_inches='tight')
plt.show()